# 三种异步调用：`ainvoke`、`astream`、`abatch`

LangChain 的许多核心对象都实现了统一的 `Runnable` 接口，例如 Chat Model、Prompt、Retriever、Chain、Graph 等。同步调用里常见的 `invoke()`、`stream()`、`batch()`，在异步场景下分别对应 `ainvoke()`、`astream()`、`abatch()`。官方文档中也说明，LangChain 采用在同步方法名前加 `a` 的约定来表示 async 版本，例如 `ainvoke`、`astream`。

异步调用的核心价值不是让单次模型推理本身变快，而是让 I/O 密集型工作不阻塞当前事件循环。LLM 应用通常会等待模型服务、数据库、向量库、HTTP API 或工具执行结果；这些等待时间可以通过 `async/await` 让给其他任务，从而提升吞吐量、响应性和并发处理能力。

## 总览对比

| 方法 | 输入规模 | 返回形式 | 消费方式 | 典型用途 |
|---|---:|---|---|---|
| `ainvoke()` | 单个输入 | 单个最终结果 | `await runnable.ainvoke(input)` | 在 async 函数中完成一次普通调用 |
| `astream()` | 单个输入 | 异步迭代器，逐步产出 chunk | `async for chunk in runnable.astream(input)` | 边生成边消费，降低首屏/首 token 等待感 |
| `abatch()` | 多个独立输入 | 结果列表 | `await runnable.abatch(inputs)` | 并发处理一组互不依赖的请求 |

三者都可以接收 `config` 参数，也就是 `RunnableConfig`。常见配置包括 `tags`、`metadata`、`callbacks`、`configurable`，以及用于批量/并发场景的 `max_concurrency`。在需要链路追踪、运行时参数传递、限制并发量或接入 LangSmith 时，`config` 是非常重要的运行时控制入口。

## `ainvoke()`：异步单次调用

`ainvoke()` 是 `invoke()` 的异步版本，用来把一个输入转换成一个最终输出。它的语义与同步 `invoke()` 基本一致：等模型或 Runnable 完成完整处理后，一次性返回最终结果。对 Chat Model 来说，返回值通常是 `AIMessage`；对其他 Runnable 来说，返回值取决于该 Runnable 的输出类型。

```python
response = await model.ainvoke("介绍一下 LangChain 的 Runnable 接口")
print(response.content)
```

适合使用 `ainvoke()` 的场景：

- 当前代码已经处在 `async def`、FastAPI/WebSocket、异步任务队列、LangGraph async node 等异步上下文中。
- 你只需要一次完整结果，不需要逐 token 或逐事件显示。
- 你希望多个上游/下游 I/O 任务可以通过 `asyncio.gather()` 等方式并发等待。

需要注意：在普通 `.py` 脚本顶层不能直接写 `await`，通常需要包在 `async def main()` 里再用 `asyncio.run(main())` 启动；在 Jupyter Notebook 中，顶层通常可以直接使用 `await`。

## `astream()`：异步流式调用

`astream()` 是 `stream()` 的异步版本，用于在结果仍在生成时逐步消费输出。它返回的是 `AsyncIterator`，所以必须使用 `async for` 遍历。对支持流式输出的 Chat Model 来说，通常会逐步产出 `AIMessageChunk`，每个 chunk 表示当前已生成内容的一部分；这些 chunk 可以实时打印、推送到前端，或者累加成一个完整消息。

```python
full = None

async for chunk in model.astream("用三句话解释 RAG"):
    print(chunk.text, end="", flush=True)
    full = chunk if full is None else full + chunk

print(full.content_blocks)
```

`astream()` 的主要价值是改善用户体验：用户不必等完整回答生成结束后才看到内容，而是可以在模型持续生成时逐步看到输出。它特别适合聊天 UI、长文本生成、Agent 过程展示、WebSocket/SSE 推送等场景。

需要注意的是，流式能力必须由链路上的组件共同支持。LangChain 的默认 `astream()` 实现可以退化为调用 `ainvoke()`，这意味着如果某个 Runnable 没有真正实现原生 streaming，它可能只会在最终结果完成后产出一次，而不是逐 token 产出。判断是否有真实流式体验，不能只看方法是否存在，还要看底层模型、Runnable 组合和输出解析器是否能处理流式 chunk。

## `abatch()`：异步批量并发调用

`abatch()` 是 `batch()` 的异步版本，用来处理一组彼此独立的输入。它接收 `list[Input]`，最终返回 `list[Output]`。默认实现会并发运行多次 `ainvoke()`，通常适合模型调用、检索、外部 API 请求等 I/O 密集型 Runnable。

```python
inputs = [
    "翻译成英文：春天来了",
    "翻译成英文：夏天很热",
    "翻译成英文：秋天落叶",
    "翻译成英文：冬天下雪",
]

responses = await model.abatch(
    inputs,
    config={"max_concurrency": 2},
)

for response in responses:
    print(response.content)
```

`abatch()` 默认会等整批任务都完成后再返回结果列表，并且结果顺序与输入顺序对应。也就是说，哪怕第 3 个输入先完成，它在最终列表中仍然位于第 3 个位置。如果希望哪个先完成就先处理哪个，可以使用对应的 `abatch_as_completed()`，它会异步产出 `(index, result)`，其中 `index` 用来映射回原始输入。

`abatch()` 还有两个非常实用的控制点：

- `config={"max_concurrency": n}`：限制同一批次中最多同时运行多少个调用，避免触发 provider 限流、占满连接池，或在本地造成过高资源压力。
- `return_exceptions=True`：让单个输入的异常作为结果返回，而不是让整批调用直接抛错；适合批处理任务中做容错和失败项重试。

需要区分的是：LangChain 文档里的 `batch()`/`abatch()` 通常指客户端侧并发调度，它和 OpenAI、Anthropic 等模型服务商提供的离线 Batch API 不是同一个概念。前者是在当前程序中并发发起多次调用，适合在线请求和中小规模批处理；后者通常是服务商侧的异步离线任务，适合大规模、可延迟处理的作业。

## 实践选择

- 只要一个最终回答：优先用 `await model.ainvoke(...)`。
- 想把生成过程实时展示出来：用 `async for chunk in model.astream(...)`。
- 有多个互不依赖的输入要一起处理：用 `await model.abatch(...)`，并用 `max_concurrency` 控制并发。
- 想批量处理但又希望先完成先消费：用 `abatch_as_completed()`。

## 参考官方文档

- LangChain Python Models 文档：`invoke()`、`stream()`、`batch()` 的调用语义与批量并发说明。https://docs.langchain.com/oss/python/langchain/models
- LangChain/Deep Agents 生产化文档：异步方法采用 `a` 前缀命名，LLM 应用通常是 I/O 密集型，异步可改善吞吐量与响应性。https://docs.langchain.com/oss/python/deepagents/going-to-production#async
- LangGraph Checkpointers 文档：异步图执行会使用 `.ainvoke`、`.astream`、`.abatch` 等异步执行入口。https://docs.langchain.com/oss/python/langgraph/checkpointers#checkpointer-interface
